# 未评分实验：决策树

在本notebook中，你将可视化决策树如何使用信息增益进行分裂。

我们将重新审视视频讲座中使用的数据集。数据集如下：

正如你在讲座中看到的，在决策树中，我们通过观察分裂带来的**信息增益**来决定是否分裂节点。（视频信息增益图片）

其中

$$\text{Information Gain} = H(p_1^\text{node})- \left(w^{\text{left}}H\left(p_1^\text{left}\right) + w^{\text{right}}H\left(p_1^\text{right}\right)\right),$$

其中 $H$ 是熵，定义为

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$

记住这里的对数是以2为底的。运行下面的代码块，亲自观察熵 $H(p)$ 随 $p$ 变化的行为。

注意，当 $p = 0.5$ 时，H 达到最大值。这意味着事件的概率为 $0.5$。而当 $p = 0$ 和 $p = 1$ 时，即事件发生的概率完全可预测时，H 达到最小值。因此，熵表示事件的可预测程度。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import *

In [2]:
%matplotlib widget
_ = plot_entropy()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

|                                                     |   Ear Shape | Face Shape | Whiskers |   Cat  |
|:---------------------------------------------------:|:---------:|:-----------:|:---------:|:------:|
| <img src="images/0.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/1.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Present  |    1   |
| <img src="images/2.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |
| <img src="images/3.png" alt="drawing" width="50"/> |   Pointy   |  Not Round  |  Present  |    0   |
| <img src="images/4.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/5.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Absent   |    1   |
| <img src="images/6.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Absent   |    0   |
| <img src="images/7.png" alt="drawing" width="50"/> |   Pointy   |  Round      |  Absent   |    1   |
| <img src="images/8.png" alt="drawing" width="50"/> |    Floppy  |   Round     |  Absent   |    0   |
| <img src="images/9.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |


我们将使用**独热编码**来编码分类特征。编码方式如下：

- 耳朵形状：尖的 = 1，耷拉的 = 0
- 脸型：圆的 = 1，不圆的 = 0
- 胡须：有 = 1，没有 = 0

因此，我们有两组数据：

- `X_train`：每个样本包含3个特征：
            - 耳朵形状（尖的为1，否则为0）
            - 脸型（圆的为1，否则为0）
            - 胡须（有为1，否则为0）
            
- `y_train`：动物是否是猫
            - 如果是猫则为1
            - 否则为0

In [3]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [4]:
#例如，第一个样本
X_train[0]

array([1, 1, 1])

这意味着第一个样本具有尖耳朵、圆脸和胡须。

在每个节点上，我们计算每个特征的信息增益，然后通过比较节点的熵与两个分裂节点的加权熵，在信息增益最大的特征上分裂节点。

因此，根节点包含数据集中的所有动物。记住 $p_1^{node}$ 是根节点中正类（猫）的比例。所以

$$p_1^{node} = \frac{5}{10} = 0.5$$

现在让我们编写一个函数来计算熵。

In [5]:
def entropy(p):
    if p == 0 or p == 1:
        return 0
    else:
        return -p * np.log2(p) - (1- p)*np.log2(1 - p)
    
print(entropy(0.5))

1.0


为了说明这一点，让我们计算如果我们对每个特征分裂节点的信息增益。为此，让我们编写一些函数。

In [7]:
def split_indices(X, index_feature):
    """给定数据集和特征索引，返回两个分裂节点的索引列表，左节点包含该特征=1的动物，右节点包含该特征=0的动物
    特征索引 = 0 => 耳朵形状
    特征索引 = 1 => 脸型
    特征索引 = 2 => 胡须
    """
    left_indices = []
    right_indices = []
    for i,x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

因此，如果我们选择耳朵形状进行分裂，那么左节点中必须包含（查看上面的表格）以下索引：

$$0 \quad 3 \quad 4 \quad 5 \quad 7$$

而右节点则包含其余的索引。

In [8]:
split_indices(X_train, 0)

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])

现在我们需要另一个函数来计算分裂节点中的加权熵。正如你在视频讲座中看到的，我们必须找到：

- $w^{\text{left}}$ 和 $w^{\text{right}}$，**每个节点**中动物的比例。
- $p^{\text{left}}$ 和 $p^{\text{right}}$，**每个分裂**中猫的比例。

注意这两个定义之间的区别！！为了说明，如果我们在索引0（耳朵形状）的特征上分裂根节点，那么在左节点中，即包含动物0、3、4、5和7的节点，我们有：

$$w^{\text{left}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{left}} = \frac{4}{5}$$
$$w^{\text{right}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{right}} = \frac{1}{5}$$

In [9]:
def weighted_entropy(X,y,left_indices,right_indices):
    """
    此函数接受分裂后的数据集、我们选择分裂的索引，并返回加权熵。
    """
    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)
    
    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

In [10]:
left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

0.7219280948873623

因此，两个分裂节点中的加权熵为0.72。要计算**信息增益**，我们必须将其从我们选择分裂的节点（在这种情况下是根节点）的熵中减去。

In [11]:
def information_gain(X, y, left_indices, right_indices):
    """
    这里，X 包含节点中的元素，y 是它们对应的类别
    """
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X,y,left_indices,right_indices)
    return h_node - w_entropy

In [12]:
information_gain(X_train, y_train, left_indices, right_indices)

0.2780719051126377

现在，让我们计算如果我们对每个特征分裂根节点的信息增益：

In [13]:
for i, feature_name in enumerate(['Ear Shape', 'Face Shape', 'Whiskers']):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name}, information gain if we split the root node using this feature: {i_gain:.2f}")
    

Feature: Ear Shape, information gain if we split the root node using this feature: 0.28
Feature: Face Shape, information gain if we split the root node using this feature: 0.03
Feature: Whiskers, information gain if we split the root node using this feature: 0.12


因此，最佳的分裂特征确实是耳朵形状。运行下面的代码查看分裂效果。你不需要理解下面的代码块。

In [14]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=1, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

 Depth 0, Root: Split on feature: 0
 - Left leaf node with indices [0, 3, 4, 5, 7]
 - Right leaf node with indices [1, 2, 6, 8, 9]


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

这个过程是**递归的**，这意味着我们必须对每个节点执行这些计算，直到满足停止条件：

- 如果分裂后的树深度超过阈值
- 如果结果节点只有1个类别
- 如果分裂的信息增益低于阈值

最终的树如下所示：

In [15]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

 Depth 0, Root: Split on feature: 0
- Depth 1, Left: Split on feature: 1
  -- Left leaf node with indices [0, 4, 5, 7]
  -- Right leaf node with indices [3]
- Depth 1, Right: Split on feature: 2
  -- Left leaf node with indices [1]
  -- Right leaf node with indices [2, 6, 8, 9]


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

恭喜！你完成了这个notebook！